# 11 Enzyme docking setup

Both enzyme workflows need a docked enzyme–PHA complex PDB first: Enzyme–PHA in water (the method used for the project's research simulations): 05B → 06B, and the Optional enzyme–PHA alternative with OpenMM: 05A → 06C. This notebook prepares the PHA input for that docking.

It takes a PHA structure from the polymer benchmark (notebook 10 writes it to `examples/output/benchmark/<SYSTEM>/`) and exports the polymer only as a PDB for manual HADDOCK docking. The docked complex PDB from HADDOCK is then the input of 06B (CHARMM-GUI Solution Builder) or 06C.

Scope: locate benchmark structures, export docking PDBs, document enzyme/polymer docking jobs, and prepare files for manual HADDOCK runs. This notebook does not submit HADDOCK jobs, does not add DiffDock or RFdiffusion, and does not start enzyme-polymer MD.

## Select Polymer System

Change `polymer_system` to prepare a different benchmark oligomer.

In [1]:
from iphasimulator.workflows.md_benchmark import SYSTEMS

supported_systems = list(SYSTEMS)
polymer_system = supported_systems[0]
if polymer_system not in supported_systems:
    raise ValueError(f"Unsupported polymer system: {polymer_system}")

polymer_system

'P3HB_4'

## Locate Benchmark Structure

Notebook 10 writes the benchmark runs to `examples/output/benchmark/{polymer_system}/`. The default structure is the final production frame:

`examples/output/benchmark/{polymer_system}/gromacs/solvated_polymer/step7_production.gro`

- `GRO_NAME`: the structure file in that folder; `"step5_input.gro"` is the starting structure before MD.
- `POLYMER_RESNAME`: the polymer's residue name: `PHA` for 06A systems (as in the benchmark), `LIG` for CHARMM-GUI systems.

In [2]:
from pathlib import Path

GRO_NAME = "step7_production.gro"  # or "step5_input.gro", the starting structure before MD
POLYMER_RESNAME = "PHA"            # PHA for 06A systems, LIG for CHARMM-GUI systems

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
benchmark_root = repo_root / "examples" / "output" / "benchmark"
solvated_polymer_dir = benchmark_root / polymer_system / "gromacs" / "solvated_polymer"
final_gro_path = solvated_polymer_dir / GRO_NAME
topology_path = solvated_polymer_dir / "topol.top"
docking_output_dir = repo_root / "examples" / "output" / "docking_inputs" / polymer_system

{
    "repo_root": repo_root,
    "final_gro_path": final_gro_path,
    "final_gro_exists": final_gro_path.exists(),
    "topology_path": topology_path,
    "docking_output_dir": docking_output_dir,
}

{'repo_root': PosixPath('/Users/k20098771/opt/iPHASimulator_v2'),
 'final_gro_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/benchmark/P3HB_4/gromacs/solvated_polymer/step7_production.gro'),
 'final_gro_exists': False,
 'topology_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/benchmark/P3HB_4/gromacs/solvated_polymer/topol.top'),
 'docking_output_dir': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/docking_inputs/P3HB_4')}

## Export Polymer PDB

HADDOCK accepts PDB input. This cell writes the polymer only (residue `POLYMER_RESNAME`; no water or ions) to `examples/output/docking_inputs/{polymer_system}/`.

Before writing, it checks that the polymer is whole: every bonded heavy-atom pair must be closer than 0.3 nm. If the polymer is split across the periodic box, the cell stops; use a whole-molecule frame instead, e.g. the representative frame from notebook 08 or `gmx trjconv -pbc mol`.

In [3]:
from iphasimulator.docking_inputs import export_polymer_pdb

polymer_pdb_path = docking_output_dir / f"{polymer_system}_{final_gro_path.stem}_for_haddock.pdb"

if final_gro_path.exists():
    exported = export_polymer_pdb(final_gro_path, topology_path, polymer_pdb_path, residue_name=POLYMER_RESNAME)
    first, second, length = exported.check.longest_bond
    print(f"✓ {POLYMER_RESNAME} is whole: {exported.check.heavy_bonds} bonded heavy-atom pairs, longest {length:.3f} nm ({first}-{second})")
    print(f"✓ Wrote {exported.atoms} {POLYMER_RESNAME} atoms to {exported.pdb_path}")
else:
    print(f"✗ Missing structure: {final_gro_path}")
    print("Finish the benchmark production run (06A / notebook 10), or set GRO_NAME = \"step5_input.gro\" to export the starting structure.")

✗ Missing structure: /Users/k20098771/opt/iPHASimulator_v2/examples/output/benchmark/P3HB_4/gromacs/solvated_polymer/step7_production.gro
Finish the benchmark production run (06A / notebook 10), or set GRO_NAME = "step5_input.gro" to export the starting structure.


## Enzyme Information

| Enzyme | Length | Catalytic triad | P3HO binding | P3HB binding | Notes |
| --- | ---: | --- | --- | --- | --- |
| GK13 | 245 aa | Ser139 / Asp195 / His227 | binds P3HO | does not bind P3HB | Benchmark enzyme for P3HO recognition |
| Anc45 | 244 aa | Ser138 / Asp194 / His226 | binds P3HO | does not bind P3HB | About 20x stronger than GK13 for P3HO |

In [4]:
enzymes = [
    {
        "enzyme": "GK13",
        "length": "245 aa",
        "catalytic_triad": "Ser139 / Asp195 / His227",
        "binds": "P3HO",
        "does_not_bind": "P3HB",
    },
    {
        "enzyme": "Anc45",
        "length": "244 aa",
        "catalytic_triad": "Ser138 / Asp194 / His226",
        "binds": "P3HO",
        "does_not_bind": "P3HB",
        "note": "About 20x stronger than GK13 for P3HO",
    },
]

enzymes

[{'enzyme': 'GK13',
  'length': '245 aa',
  'catalytic_triad': 'Ser139 / Asp195 / His227',
  'binds': 'P3HO',
  'does_not_bind': 'P3HB'},
 {'enzyme': 'Anc45',
  'length': '244 aa',
  'catalytic_triad': 'Ser138 / Asp194 / His226',
  'binds': 'P3HO',
  'does_not_bind': 'P3HB',
  'note': 'About 20x stronger than GK13 for P3HO'}]

## Manual HADDOCK Job Records

Use the exported polymer PDB as the ligand input for manual HADDOCK setup. Add enzyme PDB paths after preparing the enzyme structures outside this notebook.

In [5]:
docking_jobs = [
    {
        "job_name": f"{enzyme['enzyme']}_{polymer_system}_manual_haddock",
        "enzyme": enzyme["enzyme"],
        "enzyme_pdb": None,
        "polymer_system": polymer_system,
        "polymer_pdb": polymer_pdb_path,
        "haddock_submission": "manual",
    }
    for enzyme in enzymes
]

docking_jobs

[{'job_name': 'GK13_P3HB_4_manual_haddock',
  'enzyme': 'GK13',
  'enzyme_pdb': None,
  'polymer_system': 'P3HB_4',
  'polymer_pdb': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/docking_inputs/P3HB_4/P3HB_4_step7_production_for_haddock.pdb'),
  'haddock_submission': 'manual'},
 {'job_name': 'Anc45_P3HB_4_manual_haddock',
  'enzyme': 'Anc45',
  'enzyme_pdb': None,
  'polymer_system': 'P3HB_4',
  'polymer_pdb': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/docking_inputs/P3HB_4/P3HB_4_step7_production_for_haddock.pdb'),
  'haddock_submission': 'manual'}]

## HADDOCK Preparation Checklist

- Confirm that `GRO_NAME` (by default `step7_production.gro`, the final production frame) is the intended structure for the selected polymer.
- Inspect the exported polymer PDB before upload.
- Prepare enzyme PDB files for GK13 and Anc45 separately.
- In HADDOCK, upload enzyme as receptor and polymer as ligand.
- Record active/passive residues and run identifiers outside this notebook or in the `docking_jobs` table above.
- Do not start enzyme-polymer MD from these docking inputs until docking poses have been reviewed.
- The reviewed docked complex PDB is the input of 06B (CHARMM-GUI Solution Builder) or 06C.